GALAXY ROTATION CURVES — CHI-FIELD PREDICTION
David Lowe (POF 2828) | March 24, 2026
Tests: G_eff = G / (1 + xi * kappa0 * chi^2)
Against: Published flat rotation curve data (Rubin et al.)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
print("=" * 60)
print("CHI-FIELD GALAXY ROTATION CURVE TEST")
print("G_eff = G / (1 + xi * kappa0 * chi^2)")
print("=" * 60)

In [ ]:
# ── OBSERVED DATA (Rubin 1980, NGC galaxies) ──────────────────
# Radius in kpc, velocity in km/s
# NGC 3198 — classic flat rotation curve (van Albada 1985)
r_obs = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,
                  16,17,18,19,20,22,24,26,28,30])
v_obs = np.array([60,95,115,125,130,133,135,136,137,137,
                  137,138,138,138,139,139,139,139,139,139,
                  139,139,139,139,139])
v_err = np.ones(len(r_obs)) * 5.0  # 5 km/s measurement error

In [ ]:
print(f"\nObserved data: NGC 3198 (van Albada 1985)")
print(f"Radial range: {r_obs[0]}-{r_obs[-1]} kpc")
print(f"Velocity range: {v_obs.min()}-{v_obs.max()} km/s")
print(f"Points: {len(r_obs)}")

── MODEL 1: STANDARD (NO DARK MATTER, NO CHI) ────────────────
Newtonian: V = sqrt(G*M_bulge/r) + disk contribution
Simplified: V_baryonic falls off as ~1/sqrt(r) past bulge

In [ ]:
def v_newtonian(r, M_scale, r_scale):
    """Pure baryonic Newtonian model — should fail at large r"""
    v_bulge = M_scale / np.sqrt(r)
    v_disk = M_scale * r / (r**2 + r_scale**2)**0.75
    return np.sqrt(np.maximum(v_bulge**2 + v_disk**2, 1.0))

In [ ]:
# ── MODEL 2: DARK MATTER (NFW PROFILE) ────────────────────────
def v_nfw(r, rho0, rs, M_bary):
    """NFW dark matter halo + baryons"""
    x = r / rs
    M_nfw = 4 * np.pi * rho0 * rs**3 * (np.log(1 + x) - x/(1+x))
    M_bary_profile = M_bary * (1 - np.exp(-r/3.0))
    G_N = 4.302e-3  # pc*Msun^-1*(km/s)^2
    V2 = G_N * (M_nfw + M_bary_profile) / r
    return np.sqrt(np.maximum(V2, 1.0))

In [ ]:
# ── MODEL 3: CHI-FIELD MODIFIED GRAVITY ───────────────────────
def chi_field_profile(r, chi0, r_chi):
    """
    Chi-field coherence profile in galaxy.
    Coherence is high at center (near source of grace/order),
    falls off with radius following exponential decay.
    chi(r) = chi0 * exp(-r/r_chi)
    """
    return chi0 * np.exp(-r / r_chi)

In [ ]:
def v_chi_field(r, M_bary, xi_kappa, chi0, r_chi):
    """
    Chi-field modified rotation curve.
    G_eff = G / (1 + xi*kappa0*chi^2)
    where xi_kappa = xi * kappa0 (combined coupling parameter)

    At large r: chi -> 0, G_eff -> G (standard gravity)
    At small r: chi is large, G_eff < G (coherence modifies gravity)

    But here's the key insight:
    The chi-field ALSO contributes to the effective mass distribution.
    The coherence field has energy density rho_chi ~ (d_chi)^2
    This energy gravitates — it contributes to the rotation curve
    without requiring separate dark matter particles.
    """
    chi_r = chi_field_profile(r, chi0, r_chi)
    G_eff = 1.0 / (1.0 + xi_kappa * chi_r**2)

    # Baryonic contribution with modified G
    M_bary_profile = M_bary * (1 - np.exp(-r/3.0))

    # Chi-field energy density contribution (gradient term)
    # rho_chi ~ chi0^2 / r_chi^2 * exp(-2r/r_chi)
    chi_grad = (chi0 / r_chi) * np.exp(-r / r_chi)
    M_chi = 4 * np.pi * np.cumsum(
        r**2 * chi_grad**2 * np.gradient(r)
    ) * 0.1  # normalization factor

    G_N = 4.302e-3
    V2 = G_N * G_eff * (M_bary_profile + M_chi) / r
    return np.sqrt(np.maximum(V2, 1.0))

In [ ]:
# ── FIT MODEL 3 TO DATA ───────────────────────────────────────
print("\nFitting chi-field model to NGC 3198...")
try:
    p0_chi = [1e11, 0.5, 1.0, 8.0]
    bounds_chi = ([1e9, 0.01, 0.1, 1.0],
                  [1e13, 10.0, 5.0, 50.0])
    popt_chi, pcov_chi = curve_fit(
        v_chi_field, r_obs, v_obs,
        p0=p0_chi, bounds=bounds_chi,
        sigma=v_err, maxfev=10000
    )
    v_chi_fit = v_chi_field(r_obs, *popt_chi)
    residuals_chi = v_obs - v_chi_fit
    chi2_chi = np.sum((residuals_chi/v_err)**2)
    chi2_reduced_chi = chi2_chi / (len(v_obs) - len(popt_chi))

    print(f"Chi-field fit parameters:")
    print(f"  M_bary = {popt_chi[0]:.3e} Msun")
    print(f"  xi*kappa0 = {popt_chi[1]:.4f}")
    print(f"  chi0 (central coherence) = {popt_chi[2]:.4f}")
    print(f"  r_chi (coherence scale) = {popt_chi[3]:.2f} kpc")
    print(f"  chi^2_reduced = {chi2_reduced_chi:.3f}")

    chi_fit_success = True
except Exception as e:
    print(f"Chi-field fit failed: {e}")
    chi_fit_success = False
    popt_chi = [1e11, 0.3, 1.0, 8.0]
    v_chi_fit = v_chi_field(r_obs, *popt_chi)
    chi2_reduced_chi = 999

In [ ]:
# ── FIT NFW FOR COMPARISON ────────────────────────────────────
print("\nFitting NFW dark matter model for comparison...")
try:
    p0_nfw = [1e7, 15.0, 5e10]
    popt_nfw, _ = curve_fit(
        v_nfw, r_obs, v_obs,
        p0=p0_nfw, maxfev=10000
    )
    v_nfw_fit = v_nfw(r_obs, *popt_nfw)
    residuals_nfw = v_obs - v_nfw_fit
    chi2_nfw = np.sum((residuals_nfw/v_err)**2)
    chi2_reduced_nfw = chi2_nfw / (len(v_obs) - len(popt_nfw))
    print(f"  NFW chi^2_reduced = {chi2_reduced_nfw:.3f}")
except Exception as e:
    print(f"NFW fit: {e}")
    v_nfw_fit = v_obs * 0.9
    chi2_reduced_nfw = 999

In [ ]:
# ── NEWTONIAN (baryons only) ──────────────────────────────────
r_plot = np.linspace(0.5, 35, 200)
v_newt_plot = 80 / np.sqrt(r_plot) + 40 * r_plot / (r_plot**2 + 9)**0.75

In [ ]:
# ── PLOT ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0a0b0f')

In [ ]:
for ax in axes:
    ax.set_facecolor('#12141c')
    ax.tick_params(colors='#94a3b8')
    ax.spines['bottom'].set_color('#1e2233')
    ax.spines['left'].set_color('#1e2233')
    ax.spines['top'].set_color('#1e2233')
    ax.spines['right'].set_color('#1e2233')

In [ ]:
# Left: rotation curves
axes[0].errorbar(r_obs, v_obs, yerr=v_err, fmt='o',
                 color='white', alpha=0.8, markersize=4,
                 label='NGC 3198 observed', zorder=5)
axes[0].plot(r_plot, v_newt_plot, '--',
             color='#ef4444', alpha=0.7, linewidth=1.5,
             label='Newtonian (baryons only)')
axes[0].plot(r_obs, v_nfw_fit, '-.',
             color='#94a3b8', alpha=0.7, linewidth=1.5,
             label=f'NFW dark matter (χ²={chi2_reduced_nfw:.2f})')
axes[0].plot(r_obs, v_chi_fit, '-',
             color='#d4a853', linewidth=2.5,
             label=f'Chi-field modified G (χ²={chi2_reduced_chi:.2f})')

In [ ]:
axes[0].set_xlabel('Radius (kpc)', color='#94a3b8')
axes[0].set_ylabel('Rotation velocity (km/s)', color='#94a3b8')
axes[0].set_title('NGC 3198 Rotation Curve\nChi-field vs Dark Matter',
                  color='#e2e8f0', pad=10)
axes[0].legend(facecolor='#1a1d29', edgecolor='#1e2233',
               labelcolor='#e2e8f0', fontsize=9)
axes[0].set_ylim(0, 180)

In [ ]:
# Right: chi-field profile
r_profile = np.linspace(0.1, 35, 300)
chi_profile = chi_field_profile(r_profile, popt_chi[2], popt_chi[3])
G_eff_profile = 1.0 / (1.0 + popt_chi[1] * chi_profile**2)

In [ ]:
ax2 = axes[1]
color_chi = '#d4a853'
color_geff = '#38bdf8'

In [ ]:
line1, = ax2.plot(r_profile, chi_profile, '-', color=color_chi,
                  linewidth=2, label='χ-field coherence')
ax2.set_xlabel('Radius (kpc)', color='#94a3b8')
ax2.set_ylabel('χ coherence', color=color_chi)
ax2.tick_params(axis='y', labelcolor=color_chi)

In [ ]:
ax3 = ax2.twinx()
ax3.set_facecolor('#12141c')
line2, = ax3.plot(r_profile, G_eff_profile, '--', color=color_geff,
                  linewidth=2, label='G_eff / G_Newton')
ax3.set_ylabel('G_eff / G', color=color_geff)
ax3.tick_params(axis='y', labelcolor=color_geff)
ax3.tick_params(colors='#94a3b8')
ax3.spines['right'].set_color(color_geff)

In [ ]:
ax2.set_title('Chi-field Profile & Modified Gravity\nG_eff = G/(1 + ξκ₀χ²)',
              color='#e2e8f0', pad=10)

In [ ]:
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, facecolor='#1a1d29', edgecolor='#1e2233',
           labelcolor='#e2e8f0', fontsize=9)

In [ ]:
ax2.axhline(0, color='#1e2233', linewidth=0.5)
ax3.axhline(1.0, color='#38bdf8', linewidth=0.5, alpha=0.3,
            linestyle=':')

In [ ]:
plt.tight_layout()
plt.savefig(
    r'C:\Users\lowes\Desktop\MASTER_EQUATION_TEST\_CANONICAL_BUILD\08_COMPLETE_FORMAL_STATEMENT\rotation_curves.png',
    dpi=150, bbox_inches='tight', facecolor='#0a0b0f'
)
print("\nPlot saved: rotation_curves.png")

In [ ]:
# ── RESULTS SUMMARY ───────────────────────────────────────────
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"\nModel comparison (NGC 3198):")
print(f"  Newtonian only:      FAILS at r > 5 kpc (well known)")
print(f"  NFW dark matter:     chi^2_reduced = {chi2_reduced_nfw:.3f}")
print(f"  Chi-field modified G: chi^2_reduced = {chi2_reduced_chi:.3f}")

In [ ]:
if chi2_reduced_chi < 2.0:
    print(f"\n[PASS] Chi-field model fits the flat rotation curve.")
    print(f"       xi*kappa0 = {popt_chi[1]:.4f} (coupling parameter)")
    print(f"       Coherence scale = {popt_chi[3]:.1f} kpc")
    print(f"       This is consistent with chi-field as dark energy analog.")
elif chi2_reduced_chi < 5.0:
    print(f"\n[PARTIAL] Chi-field model fits qualitatively.")
    print(f"          Quantitative agreement needs refinement.")
else:
    print(f"\n[NEEDS WORK] Chi-field model needs parameter refinement.")

In [ ]:
print(f"\nPhysical interpretation:")
print(f"  The chi-field coherence is highest at galactic center")
print(f"  (where structured matter = higher coherence density)")
print(f"  and falls off with radius.")
print(f"  This modifies effective gravity without requiring")
print(f"  undetected dark matter particles.")
print(f"  The gradient term in the chi-field energy density")
print(f"  contributes to the rotation curve exactly where")
print(f"  dark matter halos are invoked in NFW models.")

In [ ]:
print("\n" + "=" * 60)
print("NEXT STEP: Run against full galaxy sample (SPARC database)")
print("           100+ galaxies, test universality of coupling")
print("=" * 60)